# cAPTure: XGB-P+T context ablations

This CPU notebook trains only the `current_window` and `history` context variants. It reuses the completed XGB-P, full XGB-P+T, prepared packet, fold-preprocessing, and context artifacts. The five approved development scenarios and their original out-of-background folds remain unchanged. No threshold is selected and no final-test data is read.


## 1. Prepare the Colab environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/requirements-capture-xgb.txt",
                  "code/python/utils/capture_xgb_p_t.py",
                  "configs/capture_experiment_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


## 2. Bind immutable primary artifacts

Set `PRIMARY_RUN_ID` to the completed XGB-P+T run ID shown by the primary notebook. Set `ABLATION_RUN_ID` only when resuming an existing ablation run. A new ID creates new output directories; incomplete directories are never overwritten automatically.


In [ ]:
from utils.capture_xgb_p import summarize_xgb_p_oof
from utils.capture_xgb_p_t import (
    context_features_for_variant, run_xgb_p_t_fold, summarize_xgb_p_t_oof,
    validate_context_run, validate_xgb_p_t_fold_run,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs/20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs/20260919T004143_161396Z_preprocessing"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
PRIMARY_RUN_ID = None  # Set to the completed XGB-P+T run ID.
ABLATION_RUN_ID = None  # Set to an existing ablation run ID only when resuming.
if PRIMARY_RUN_ID is None:
    raise ValueError("Set PRIMARY_RUN_ID to the completed XGB-P+T run ID.")
if ABLATION_RUN_ID is None:
    ABLATION_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p_t_ablation"
CONTEXT_DIR = DRIVE_ROOT / "xgb_p_t_context_runs" / PRIMARY_RUN_ID
FULL_RUN_DIR = DRIVE_ROOT / "xgb_p_t_runs" / PRIMARY_RUN_ID
ABLATION_DIR = DRIVE_ROOT / "xgb_p_t_ablation_runs" / ABLATION_RUN_ID
LOCAL_WORK_ROOT = Path("/content/capture_xgb_p_t_ablation_work")
BATCH_SIZE = 50_000
NTHREAD = 2
context_report = validate_context_run(
    manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH,
    prepared_run_dir=PREPARED_RUN_DIR, context_dir=CONTEXT_DIR)
full_summary = summarize_xgb_p_t_oof(FULL_RUN_DIR)
baseline_summary = summarize_xgb_p_oof(BASELINE_RUN_DIR, "depth5_primary")
print("Context scenarios:", list(context_report["scenarios"]))
print("Ablation output:", ABLATION_DIR)
print("Full-model macro ROC-AUC:", full_summary["hierarchical_macro_oof_packet_roc_auc"])


## 3. Train the current-window variant

The model has the same 103 packet columns plus the eight complete-current-window fields. Train fold A first, then fold B. Each fold fits its own context scaler using its training scenarios only.


In [ ]:
def train_or_verify(variant_name, fold):
    output_dir = ABLATION_DIR / variant_name / "depth5_primary" / f"fold_{fold}"
    if output_dir.exists():
        return validate_xgb_p_t_fold_run(output_dir, fold, variant_name)
    return run_xgb_p_t_fold(
        manifest_path=MANIFEST_PATH,
        packet_schema_path=PACKET_SCHEMA_PATH,
        preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
        prepared_run_dir=PREPARED_RUN_DIR,
        preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR,
        context_dir=CONTEXT_DIR,
        output_dir=output_dir,
        local_work_root=LOCAL_WORK_ROOT,
        fold=fold,
        variant_name=variant_name,
        batch_size=BATCH_SIZE,
        nthread=NTHREAD,
    )

current_a = train_or_verify("current_window", "A")
display(pd.DataFrame.from_dict(current_a["validation"], orient="index")[[
    "rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])


In [ ]:
current_b = train_or_verify("current_window", "B")
display(pd.DataFrame.from_dict(current_b["validation"], orient="index")[[
    "rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])


## 4. Train the preceding-history variant

The model has the same 103 packet columns plus the six features from the preceding 30 seconds. The current five-second window is excluded from these six features.


In [ ]:
history_a = train_or_verify("history", "A")
display(pd.DataFrame.from_dict(history_a["validation"], orient="index")[[
    "rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])


In [ ]:
history_b = train_or_verify("history", "B")
display(pd.DataFrame.from_dict(history_b["validation"], orient="index")[[
    "rows", "packet_roc_auc", "packet_pr_auc_diagnostic"]])


## 5. Compare the four development OOF rankings

These rankings are diagnostic. Use the separate operational OOF notebook for false-alert budgets, attack-step coverage, and latency.


In [ ]:
current_summary = summarize_xgb_p_t_oof(
    ABLATION_DIR / "current_window", "current_window")
history_summary = summarize_xgb_p_t_oof(
    ABLATION_DIR / "history", "history")
summaries = {
    "xgb_p": baseline_summary,
    "current_window": current_summary,
    "history": history_summary,
    "full": full_summary,
}
rows = []
for scenario, baseline in baseline_summary["scenario_metrics"].items():
    row = {"scenario": scenario, "fold": baseline["fold"],
           "packets": baseline["packets"]}
    for name, summary in summaries.items():
        row[f"{name}_packet_roc_auc"] = summary["scenario_metrics"][scenario]["packet_roc_auc"]
    rows.append(row)
display(pd.DataFrame(rows).set_index("scenario"))
display(pd.DataFrame([{
    "model": name,
    "hierarchical_macro_oof_packet_roc_auc": summary["hierarchical_macro_oof_packet_roc_auc"],
} for name, summary in summaries.items()]))
print("Ablation run ID:", ABLATION_RUN_ID)
